In [13]:
import json
import pandas as pd
import random
# Load the two JSON files
file_direct = "../data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json"
file_cot = "../data/GSM8K/output/output.GSM8K.cot0shot.math_teacher.llama2-7b-chat.json"

with open(file_direct, "r") as f:
    data_direct = json.load(f)

with open(file_cot, "r") as f:
    data_cot = json.load(f)

# Convert to dictionaries indexed by question ID for easy comparison
dict_direct = {entry["id"]: entry for entry in data_direct}
dict_cot = {entry["id"]: entry for entry in data_cot}

# 1. Find the calibration set: CoT result is True and Direct result is False
calibration_set = []
for qid in dict_direct:
    if qid in dict_cot:
        if dict_cot[qid].get("cot0shot.math teacher_result") and not dict_direct[qid].get("direct.math teacher_result"):
            calibration_set.append(qid)

# 2. Question IDs of the calibration set
calibration_ids = calibration_set

# 3. choose these calibration_ids samples from these two files and save them
calibration_data_cot = [dict_cot[qid] for qid in calibration_ids]   
with open("../data/GSM8K/output/calibration_set_cot.json", "w") as f:
    json.dump(calibration_data_cot, f, indent=4)

direct_true_ids = [qid for qid, entry in dict_direct.items() if entry.get("direct.math teacher_result") is True]
direct_true_samples = [dict_direct[qid] for qid in direct_true_ids]

# 4. Calibration set of direct should be all direct.math teacher_resul true samples plus some other samples
# find the true samples in direct
num_extra = 120 - len(direct_true_ids)  # number of extra samples needed
calibration_data_direct = [dict_direct[qid] for qid in calibration_ids]
calibration_candidates = [entry for entry in calibration_data_direct if entry["id"] not in direct_true_ids]
extra_samples = random.sample(calibration_candidates, min(num_extra, len(calibration_candidates)))

direct_calibration_samples = direct_true_samples + extra_samples

with open("../data/GSM8K/output/calibration_set_direct_mixed.json", "w") as f:
    json.dump(direct_calibration_samples, f, indent=4)


In [9]:
len(direct_true_samples)

30

In [4]:
len(calibration_ids)

120

In [ ]:

# # 3. randomly choose 100 samples not in the calibration set as the evaluation set
# /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/test.jsonl
evaluation_ids = [qid for qid in dict_direct if qid not in calibration_ids][:100]



In [ ]:
# 1. find the difference sets of these two files as the calibration set
"cot0shot.math teacher_result": true while "direct.math teacher_result": false
# 2. the question id of the calibration sets
# 3. find 100 sample set except the calibration set as the evaluation set

In [ ]:
alignment-attribution-code/data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json 
alignment-attribution-code/data/GSM8K/output/output.GSM8K.cot0shot.math_teacher.llama2-7b-chat.json